# 🕺 Dancing Stick Figures v0.3 — train an image DiT, then a video DiT

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sprited-ai/dancing-stick-figures/blob/main/notebooks/dancing_stick_figures_colab_v0_3.ipynb)

*A hands-on notebook for readers with basic Python and neural-network familiarity. The reference `64²` route is designed for one conventional GPU; `32²` is a faster sanity check.*

**What you'll do**
1. 👀 Inspect the videos and their hidden skeleton labels.
2. 🎨 Train a small **image DiT** from noise.
3. 🎬 Reuse those weights in a **video DiT** that jointly generates a 3.2-second action at the native 20 fps.
4. 🤖 Generate from your own prompt and diagnose visible structural failures.

The released clips contain 120 frames at 20 fps. The teaching protocol trains on the first 64 frames of each clip (3.2 seconds at the native 20 fps): the source motions concentrate their prompted action early, so the later frames often continue or idle. Native dataset evaluation still uses the complete 120-frame clips.

Before you start: **Runtime → Change runtime type → GPU** (T4 is sufficient for the conservative batch below).

In [ ]:
#@title 0. Setup (≈2 min) — grab the code and the small version of the dataset
import os, sys, subprocess, glob, time
V03_STARTED = time.time()
if not os.path.exists("dancing-stick-figures"):
    !git clone -q https://github.com/sprited-ai/dancing-stick-figures
%cd dancing-stick-figures
!pip install -q -r train/requirements.txt 2>&1 | tail -1
DATA_DOWNLOAD_ATTEMPTS = 3
download_cmds = [
    ["hf", "download", "sprited/dancing-stick-figures", "--repo-type", "dataset", "--include", pattern, "--local-dir", "data/hf"]
    for pattern in ("mini/*", "motion/val-*")
]
for attempt in range(1, DATA_DOWNLOAD_ATTEMPTS + 1):
    results = [subprocess.run(cmd, text=True, capture_output=True) for cmd in download_cmds]
    mini_files = glob.glob("data/hf/mini/*.parquet"); motion_files = glob.glob("data/hf/motion/*.parquet")
    if all(result.returncode == 0 for result in results) and mini_files and motion_files: break
    print(f"dataset download attempt {attempt} did not finish; retrying")
else:
    raise RuntimeError("Dataset download did not produce mini parquet files after three attempts")
print(len(mini_files), "mini shards,", len(motion_files), "motion shard(s)")
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — switch the runtime to GPU!")

#@title 0b. Choose a resolution
IMAGE_SIZE = "64" #@param ["32", "64"]
IMAGE_SIZE = int(IMAGE_SIZE)
VIDEO_FRAMES, FRAME_STRIDE = 64, 1
IMAGE_BATCH = 128 if IMAGE_SIZE == 32 else 64
VIDEO_BATCH = 8 if IMAGE_SIZE == 32 else 4
RUN_TAG = f"{IMAGE_SIZE}px"
IMAGE_RUN, VIDEO_RUN = f"runs/dit_img_{RUN_TAG}", f"runs/dit_vid64_{RUN_TAG}"
print(f"resolution={IMAGE_SIZE}² · factorised DiT · {VIDEO_FRAMES} frames at 20 fps")
print("32² is a faster sanity check; 64² is the reference setting.")

## 🧩 The fixed reference backbone

The same **factorised video DiT** is used for both stages. At `T=1`, it is an image generator. At `T=64`, it becomes a video generator.

```text
noisy RGBA frames [4, T, size, size]
             │
             ▼  split each frame into 4×4 patches
spatial attention: patches within the same frame exchange information
temporal attention: the same patch position exchanges information across all frames
text cross-attention: the complete prompt conditions every block
             │
             ▼
predicted flow from noise toward clean video [4, T, size, size]
```

Spatial and temporal attention alternate. This makes the division of labour visible in code and lets each generated time point use both earlier and later context inside the 3.2-second training window. A frozen T5-small encoder supplies prompt tokens; the DiT itself learns the rendered figure and motion from this dataset.

The exercise fixes the architecture and exposes its source. The only initial choice is `32²` for a quick sanity check or `64²` for the main run.

In [ ]:
#@title Read the actual backbone and image-to-video initialization code (optional)
import inspect
from train.video_dit_fm import Attention, TextCrossAttention, Block, VideoDiT, prepare_warmstart_state
for component in (Attention, TextCrossAttention, Block, VideoDiT, prepare_warmstart_state):
    print(f"\n# --- {component.__name__} ---")
    print(inspect.getsource(component))
print("To experiment later, edit train/video_dit_fm.py in Colab's Files pane. The lesson below uses this reference code unchanged.")

## 1 · 👀 Look at the data

Every dancer here is a **3D skeleton** (27 joints, like a video-game character) that a computer moved to a text prompt such as
*"A person does the running man dance"*, then photographed with a virtual camera. We keep the photo **and** the skeleton,
so we always know where every arm and leg *really* is. Colours tell body parts apart: red/orange = left arm, blue/cyan = right arm,
purple/pink = left leg, green = right leg. Head and body are black.

In [ ]:
#@title Contact sheet — 32 random frames with their prompts
import pyarrow.parquet as pq, glob, io, random, textwrap, numpy as np
from PIL import Image, ImageDraw
files = sorted(glob.glob("data/hf/mini/train-*.parquet")); random.seed(1)
row_counts = [pq.read_metadata(f).num_rows for f in files]
cumulative_rows = np.cumsum(row_counts); frame_candidates = range(int(cumulative_rows[-1]))
selected_frames = random.sample(frame_candidates, 32)
W, pad, lab, cols = 96, 6, 44, 8
sheet = Image.new("RGB", (cols*(W+pad)+pad, 4*(W+lab+pad)+pad), "white"); d = ImageDraw.Draw(sheet)
def over_white(b):
    im = Image.open(io.BytesIO(b)).convert("RGBA"); bg = Image.new("RGBA", im.size, "white"); bg.alpha_composite(im); return bg.convert("RGB")
tables = {}
for k, global_i in enumerate(selected_frames):
    shard_i = int(np.searchsorted(cumulative_rows, global_i, side="right"))
    local_i = global_i - (int(cumulative_rows[shard_i - 1]) if shard_i else 0)
    if shard_i not in tables: tables[shard_i] = pq.read_table(files[shard_i], columns=["color", "text", "group"])
    t = tables[shard_i]; r, c = divmod(k, cols); x = pad + c*(W+pad); y = pad + r*(W+lab+pad)
    sheet.paste(over_white(t.column("color")[local_i].as_py()["bytes"]).resize((W, W), Image.NEAREST), (x, y))
    d.text((x, y+W+2), t.column("group")[local_i].as_py(), fill="black")
    for li, line in enumerate(textwrap.wrap(t.column("text")[local_i].as_py(), 18)[:3]): d.text((x, y+W+13+10*li), line, fill=(90,90,90))
sheet

In [ ]:
#@title The hidden skeleton — draw the 27 joints on top of a frame
from IPython.display import display
t = pq.read_table(files[0], columns=["color", "joint_xy", "joint_visible", "text"])
i = 40
im = over_white(t.column("color")[i].as_py()["bytes"]).resize((256, 256), Image.NEAREST)
xy = np.frombuffer(t.column("joint_xy")[i].as_py(), np.float32).reshape(27, 2) * 256      # positions are stored as fractions of the image
vis = np.frombuffer(t.column("joint_visible")[i].as_py(), np.uint8)
sys.path.insert(0, "."); from generator.skeleton import NAMES, PARENT
dd = ImageDraw.Draw(im)
for j, n in enumerate(NAMES):
    p = PARENT.get(n)
    if p: dd.line([tuple(xy[j]), tuple(xy[NAMES.index(p)])], fill=(255,255,255), width=3); dd.line([tuple(xy[j]), tuple(xy[NAMES.index(p)])], fill=(0,0,0), width=1)
for j in range(27): x_, y_ = xy[j]; dd.ellipse([x_-3, y_-3, x_+3, y_+3], fill=(0,200,0) if vis[j] else (220,0,0))
print(t.column("text")[i].as_py(), "— green dots: joints the camera can see, red: hidden behind the body")
display(im)

In [ ]:
#@title Where did the dancer walk? — the root (hip) path of one clip, from the `motion` config
import matplotlib.pyplot as plt
if not glob.glob("data/hf/motion/val-*.parquet"):
    !hf download sprited/dancing-stick-figures --repo-type dataset --include "motion/val-*" --local-dir data/hf 2>&1 | tail -2
mo = pq.read_table(glob.glob("data/hf/motion/val-*.parquet")[0])
for k in range(4):
    r = mo.slice(k, 1).to_pylist()[0]; T = r["n_frames"]
    P = np.frombuffer(r["posed_joints"], np.float32).reshape(T, 27, 3)     # world coordinates in metres, 27 joints
    plt.plot(P[:, 0, 0], P[:, 0, 2], label=r["text"][:38])                # joint 0 = Hips, seen from above (x, z)
plt.axis("equal"); plt.xlabel("x (m)"); plt.ylabel("z (m)"); plt.title("Hip path seen from above, 6 seconds"); plt.legend(fontsize=7); plt.show()

## 2 · 🎨 Train an image model

First we unpack the frames into a fast cache. Then we train the DiT with one frame at a time. Flow matching draws a noisy point between a real frame and Gaussian noise; the model learns the direction that leads back toward the clean frame.

This stage teaches spatial structure before temporal modelling begins. Samples are saved every 500 steps so you can inspect the learning curve.

In [ ]:
#@title Unpack frames (train + validation) into a fast cache
!python -m train.cache --data data/hf/mini --out data/cache --splits train,val --threads 4 2>&1 | grep -v shards

In [ ]:
#@title Train the image DiT — 2,000 steps
import time
STEPS = 2000  #@param {type:"integer"}
image_started = time.perf_counter()
!python -m train.video_dit_fm --cache data/cache --out $IMAGE_RUN --arch dit --size $IMAGE_SIZE --frames 1 --stride 1 --first_frames 64 --batch $IMAGE_BATCH --steps $STEPS --dim 384 --depth 12 --heads 6 --patch 4 --cond text --cfg_drop 0.1 --fg_weight 2 --grad_ckpt --fast --workers 2 --sample_every 500 --val_every 500 2>&1 | grep --line-buffered -v Warning | grep --line-buffered "^cached\|^init\|^step\|wrote\|params\|Error\|Traceback"
IMAGE_WALL_SECONDS = time.perf_counter() - image_started
print(f"image stage wall time: {IMAGE_WALL_SECONDS / 60:.1f} min")

In [ ]:
#@title Look at what it learned — fixed-noise drawings at each checkpoint
from PIL import Image
for f in sorted(glob.glob(f"{IMAGE_RUN}/sample_0*.png")):
    if "raw" in f: continue
    print(f.split("/")[-1]); display(Image.open(f).resize((512, 512), Image.NEAREST))

> **Why do early samples look rough?** Two thousand steps are enough to expose the complete training loop, not to equal the released reference checkpoint. The longer reference image stage runs for 30,000 steps. Your image checkpoint is still useful: it supplies the spatial weights for the video stage below.

## 3 · 🎬 Make it move — train a 3.2-second video generator

We now build the same DiT with 64 temporal positions and initialise every compatible spatial, text, and output weight from your image model. Temporal positions and temporal attention then learn from video.

Each source clip remains available in full at 120 frames and 20 fps. For this training exercise we keep the first 64 frames — 3.2 seconds at the native 20 fps. The source motions perform their prompted action early (the generator does not pace an action to the requested duration), so this window concentrates training on the action itself. This is the protocol used by the released prompt-conditioned DiT reference.

> **Prompt conditioning.** Both stages use complete prompts through frozen T5-small token features. The model is prompt-conditioned; the notebook demonstrates sensitivity to prompt changes but does not claim a calibrated semantic adherence score.

> **Why no Video VAE?** At `32²`–`64²`, direct pixel training is practical. Keeping generated pixels in the renderer's representation avoids adding codec reconstruction errors to the first experiment.

Unlike the earlier autoregressive lesson, this baseline generates the complete clip jointly. Every temporal-attention layer can coordinate earlier and later frames at a shared patch position.

In [ ]:
#@title Train the 64-frame video DiT on top of your image DiT
INIT_CKPT = f"{IMAGE_RUN}/ckpt.pt"
assert os.path.exists(INIT_CKPT), f"Missing {INIT_CKPT}; finish image training first."
VSTEPS = 2000  #@param {type:"integer"}
video_started = time.perf_counter()
!python -m train.video_dit_fm --cache data/cache --out $VIDEO_RUN --arch dit --size $IMAGE_SIZE --frames $VIDEO_FRAMES --stride $FRAME_STRIDE --first_frames 64 --batch $VIDEO_BATCH --steps $VSTEPS --dim 384 --depth 12 --heads 6 --patch 4 --cond text --cfg_drop 0.1 --fg_weight 2 --img_frac 0.1 --i2v_frac 0.2 --init $INIT_CKPT --grad_ckpt --fast --workers 2 --sample_every $VSTEPS --val_every 500 2>&1 | grep --line-buffered -v Warning | grep --line-buffered "^cached\|^init\|^step\|wrote\|params\|Error\|Traceback"
VIDEO_WALL_SECONDS = time.perf_counter() - video_started
print(f"video stage wall time: {VIDEO_WALL_SECONDS / 60:.1f} min")

In [ ]:
#@title Type a prompt, then watch four 3.2-second samples
from pathlib import Path
from IPython.display import Image as IPImage
PROMPT = "A person runs forward."  #@param {type:"string"}
PROMPT_DIR = Path(f"out/dit_prompt_{RUN_TAG}")
PROMPT_DIR.mkdir(parents=True, exist_ok=True)
(PROMPT_DIR / "prompt.txt").write_text(PROMPT + "\n")
!python -m eval.post_eval_t2v --ckpt $VIDEO_RUN/ckpt.pt --out $PROMPT_DIR --prompts_file $PROMPT_DIR/prompt.txt --same_prompt "$PROMPT" --n 4 --steps 30 --cfg 3 --batch 1 --seed 1234 --fps 20 --strip_frames 0,21,42,63 --save_rgba 2>&1 | grep -v FutureWarning
MINE_GIF = str(PROMPT_DIR / "fixed_prompt_varied_noise_labeled.gif")
print(f"yours — prompt: {PROMPT!r}"); display(IPImage(filename=MINE_GIF))

> The image stage teaches the model how a clean figure is assembled. The video stage teaches how those parts change together over a 3.2-second window. Compare the four samples: changing noise should change the motion while preserving a coherent figure. If they remain rough, the first useful experiment is simply to train longer and compare the fixed-noise strips again.

## 4 · 🤖 Diagnose visible failures

How can we describe a visible failure without asking a human to inspect every frame? Because every body part has its own
colour, a small structural evaluator can **count**: is there exactly one red arm? Is the pink shin touching the purple thigh? Are the
colours clean or smeared? These measurements diagnose specific properties; they are not a single overall grade.

- **tvr** — *topology violation rate*: a coloured limb is missing or split into more than one piece
- **lie** — *limb-identity error*: colours that should meet (for example, upper arm and forearm) fail to touch
- **clean** — share of drawings with zero mistakes

One catch: real dancers sometimes hide an arm behind their body — the robot counts that as "missing" too! So we always
compare against **real-reference** frames. A lower score is not automatically better than the real reference: an unnaturally simple or frozen figure can also be easy for this colour-based checker.

The rollout cell also reports centroid speed, acceleration, motion fraction, and angular jerk beside the same measurements on real clips. These numbers describe different visible motion behaviours; none is a single overall quality grade.

In [ ]:
#@title Measure visible structure in your image DiT and real validation frames
import json
MINE_JSON = f"out/dit_image_{RUN_TAG}.json"
!python -m eval.score_images --ckpt $IMAGE_RUN/ckpt.pt --cache data/cache --n 128 --steps 30 --cfg 3 --out $MINE_JSON 2>&1 | tail -1
mine = json.load(open(MINE_JSON))
print(f"{'':22s} {'lie':>6s} {'tvr':>6s} {'clean':>6s}")
print(f"{'your image DiT':22s} {mine['lie']:6.3f} {mine['tvr']:6.3f} {mine['clean_frac']:6.2f}")
print(f"{'real reference':22s} {mine['floor']['lie']:6.3f} {mine['floor']['tvr']:6.3f} {mine['floor']['clean_frac']:6.2f}")

In [ ]:
#@title Verification record — timings, memory, and required artifacts
from pathlib import Path
import re
def peak_gb(path):
    values = [float(x) for x in re.findall(r"peak ([0-9.]+)GB", Path(path).read_text())]
    return max(values) if values else float("nan")
required = [f"{IMAGE_RUN}/ckpt.pt", f"{VIDEO_RUN}/ckpt.pt", MINE_GIF, MINE_JSON]
missing = [path for path in required if not Path(path).exists()]
assert not missing, f"missing required artifacts: {missing}"
verification = {
    "image_wall_seconds": IMAGE_WALL_SECONDS,
    "video_wall_seconds": VIDEO_WALL_SECONDS,
    "image_peak_gb": peak_gb(IMAGE_RUN + "/log.txt"),
    "video_peak_gb": peak_gb(VIDEO_RUN + "/log.txt"),
    "total_wall_seconds": time.time() - V03_STARTED,
    "gpu": torch.cuda.get_device_name(0),
    "resolution": IMAGE_SIZE,
    "image_steps": STEPS,
    "video_steps": VSTEPS,
    "video_frames": VIDEO_FRAMES,
    "frame_stride": FRAME_STRIDE,
}
Path("out").mkdir(exist_ok=True)
Path("out/v03_completion.json").write_text(json.dumps(verification, indent=2))
print(json.dumps(verification, indent=2))
print("V03_COMPLETE=1")

## 🚀 Where to go next

- Train the same fixed backbone longer and compare the fixed-noise strips.
- Read the printed `VideoDiT` and `Block` source to see exactly where spatial, temporal, and text attention occur.
- Change the training input or add a conditioning signal once you understand the reference run.
- Use the exact state labels for another task, such as estimating a skeleton from an image.

Dataset: https://huggingface.co/datasets/sprited/dancing-stick-figures · Code: https://github.com/sprited-ai/dancing-stick-figures · Made by Sprited.